In [1]:
%cd ..

/home/pablo/projects/nli


In [2]:
from dotenv import dotenv_values

config = dotenv_values(".env")
HF_TOKEN = config["HF_WRITE_TOKEN"]

## Counter NLI

In [10]:
from datasets import load_dataset, DatasetDict

# Define dataset names
INPUT_DATASET = "tasksource/counterfactually-augmented-snli"
OUTPUT_DATASET = "pablomiralles22/counter-nli"

# Define column mappings
MAPPINGS = {
    "sentence1": "premise",
    "sentence2": "hypothesis",
    "gold_label": "label",
}

# Load dataset
dataset = load_dataset(INPUT_DATASET)

# Rename columns for each split
if isinstance(dataset, DatasetDict):
    dataset = DatasetDict({split: ds.rename_columns(MAPPINGS) for split, ds in dataset.items()})
else:
    dataset = dataset.rename_columns(MAPPINGS)

# Map labels to integers
label_mapping = {
    "entailment": 0,
    "neutral": 1,
    "contradiction": 2,
}
dataset = dataset.map(
    lambda x: {"label": [label_mapping[label] for label in x["label"]]},
    batched=True,
    remove_columns=["label"],
)

# Push to Hugging Face Hub
dataset.push_to_hub(OUTPUT_DATASET, private=True, token=HF_TOKEN)
print(f"Dataset uploaded to {OUTPUT_DATASET}")

Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.38s/it]


Dataset uploaded to pablomiralles22/counter-nli


## HANS

In [17]:
from datasets import load_dataset, DatasetDict

# Define dataset names
OUTPUT_DATASET = "pablomiralles22/hans"

# Define column mappings
MAPPINGS = {
    "sentence1": "premise",
    "sentence2": "hypothesis",
    "gold_label": "label",
}

# Load dataset
URL = "https://raw.githubusercontent.com/tommccoy1/hans/refs/heads/master/heuristics_evaluation_set.jsonl"
dataset = load_dataset("json", data_files=URL)["train"]

# Rename columns
dataset = dataset.rename_columns(MAPPINGS)

# Map labels to integers
label_mapping = {
    "entailment": 0,
    "non-entailment": 1,
}
dataset = dataset.map(
    lambda x: {"label": [label_mapping[label] for label in x["label"]]},
    batched=True,
    remove_columns=["label"],
)

# Create split by heuristic
unique_heuristics = set(dataset["heuristic"])
print(unique_heuristics)

dataset = DatasetDict(
    {
        "test_lex": dataset.filter(lambda x: x["heuristic"] == "lexical_overlap"),
        "test_sub": dataset.filter(lambda x: x["heuristic"] == "subsequence"),
        "test_cons": dataset.filter(lambda x: x["heuristic"] == "constituent"),
    }
)

# Push to Hugging Face Hub
dataset.push_to_hub(OUTPUT_DATASET, private=True, token=HF_TOKEN)
print(f"Dataset uploaded to {OUTPUT_DATASET}")

Map: 100%|██████████| 30000/30000 [00:00<00:00, 153610.88 examples/s]


{'subsequence', 'lexical_overlap', 'constituent'}


Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.36s/it]


Dataset uploaded to pablomiralles22/hans


In [18]:
dataset

DatasetDict({
    test_lex: Dataset({
        features: ['heuristic', 'template', 'sentence1_binary_parse', 'sentence2_parse', 'subcase', 'sentence2_binary_parse', 'premise', 'hypothesis', 'sentence1_parse', 'label', 'pairID'],
        num_rows: 10000
    })
    test_sub: Dataset({
        features: ['heuristic', 'template', 'sentence1_binary_parse', 'sentence2_parse', 'subcase', 'sentence2_binary_parse', 'premise', 'hypothesis', 'sentence1_parse', 'label', 'pairID'],
        num_rows: 10000
    })
    test_cons: Dataset({
        features: ['heuristic', 'template', 'sentence1_binary_parse', 'sentence2_parse', 'subcase', 'sentence2_binary_parse', 'premise', 'hypothesis', 'sentence1_parse', 'label', 'pairID'],
        num_rows: 10000
    })
})

## NLI Diagnostics

In [24]:
from datasets import load_dataset, DatasetDict

# Define dataset names
OUTPUT_DATASET = "pablomiralles22/nli-diagnostic"

# Define column mappings
MAPPINGS = {
    "Premise": "premise",
    "Hypothesis": "hypothesis",
    "Label": "label",
}

# Load dataset
URL = "https://www.dropbox.com/s/ju7d95ifb072q9f/diagnostic-full.tsv?dl=1"
dataset = load_dataset("csv", data_files=URL, delimiter="\t")["train"]

# Rename columns
dataset = dataset.rename_columns(MAPPINGS)

# Map labels to integers
label_mapping = {
    "entailment": 0,
    "neutral": 1,
    "contradiction": 2,
}
dataset = dataset.map(
    lambda x: {"label": [label_mapping[label] for label in x["label"]]},
    batched=True,
    remove_columns=["label"],
)

# Create split
dataset = DatasetDict(
    {
        "test_know": dataset.filter(lambda x: x["Knowledge"] is not None),
        "test_logic": dataset.filter(lambda x: x["Logic"] is not None),
        "test_ls": dataset.filter(lambda x: x["Predicate-Argument Structure"] is not None),
        "test_pas": dataset.filter(lambda x: x["Lexical Semantics"] is not None),
    }
)

# Push to Hugging Face Hub
dataset.push_to_hub(OUTPUT_DATASET, private=True, token=HF_TOKEN)
print(f"Dataset uploaded to {OUTPUT_DATASET}")

dataset

Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.14s/it]


Dataset uploaded to pablomiralles22/nli-diagnostic


DatasetDict({
    test_know: Dataset({
        features: ['Lexical Semantics', 'Predicate-Argument Structure', 'Logic', 'Knowledge', 'Domain', 'premise', 'hypothesis', 'label'],
        num_rows: 284
    })
    test_logic: Dataset({
        features: ['Lexical Semantics', 'Predicate-Argument Structure', 'Logic', 'Knowledge', 'Domain', 'premise', 'hypothesis', 'label'],
        num_rows: 364
    })
    test_ls: Dataset({
        features: ['Lexical Semantics', 'Predicate-Argument Structure', 'Logic', 'Knowledge', 'Domain', 'premise', 'hypothesis', 'label'],
        num_rows: 424
    })
    test_pas: Dataset({
        features: ['Lexical Semantics', 'Predicate-Argument Structure', 'Logic', 'Knowledge', 'Domain', 'premise', 'hypothesis', 'label'],
        num_rows: 368
    })
})

## e-SNLI

In [4]:
from datasets import load_dataset, DatasetDict, concatenate_datasets

BASE_URL = "https://raw.githubusercontent.com/OanaMariaCamburu/e-SNLI/refs/heads/master/dataset/"

train_1_ds = load_dataset("csv", data_files=f"{BASE_URL}/esnli_train_1.csv")["train"]
train_2_ds = load_dataset("csv", data_files=f"{BASE_URL}/esnli_train_2.csv")["train"]
dev_ds = load_dataset("csv", data_files=f"{BASE_URL}/esnli_dev.csv")["train"]
test_ds = load_dataset("csv", data_files=f"{BASE_URL}/esnli_test.csv")["train"]

dataset = DatasetDict({
    "train": concatenate_datasets([train_1_ds, train_2_ds]),
    "dev": dev_ds,
    "test": test_ds,
})

# Define column mappings
MAPPINGS = {
    "Sentence1": "premise",
    "Sentence2": "hypothesis",
    "gold_label": "label",
    "Explanation_1": "explanation",
}
# Rename columns
dataset = dataset.rename_columns(MAPPINGS)

# Map labels to integers
label_mapping = {
    "entailment": 0,
    "neutral": 1,
    "contradiction": 2,
}
dataset = dataset.map(
    lambda x: {"label": [label_mapping[label] for label in x["label"]]},
    batched=True,
    remove_columns=["label"],
)

# Remove columns
def remove_cols(ds):
    columns_to_keep = MAPPINGS.values()
    columns_to_remove = list(set(ds.column_names) - set(columns_to_keep))
    return ds.remove_columns(columns_to_remove)

dataset = DatasetDict(
    { split: remove_cols(ds) for split, ds in dataset.items() }
)

# Push to Hugging Face Hub
dataset.push_to_hub("pablomiralles22/esnli", private=True, token=HF_TOKEN)
print(f"Dataset uploaded to pablomiralles22/esnli")

# Save to "out/data/esnli"
# dataset.save_to_disk("out/data/esnli")

Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.07s/it]


Dataset uploaded to pablomiralles22/esnli
